# ML Trading Pairs Selection via PCA + Clustering

This notebook demonstrates a method to discover candidate pairs of stocks for pairs trading using:
1. **PCA** for dimensionality reduction of asset returns
2. **OPTICS** clustering to group similar assets
3. **Cointegration + Hurst + Half-life tests** to validate candidate pairs

Source: HandsOnAITradingBook, Section 06, Example 09

> **[REFERENCE QC Cloud]**
> Ce notebook utilise QuantBook et necessite l'environnement QuantConnect Cloud.
> Pour executer : https://www.quantconnect.com/research


In [1]:
from AlgorithmImports import *
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import OPTICS
import numpy as np
import pandas as pd

## Step 1: Universe Selection and Data Loading

Select a universe of liquid US equities from an ETF (IWB = Russell 1000) and load 3 years of daily price history.

In [2]:
qb = QuantBook()

end_date = datetime(2024, 1, 3)

# Get universe constituents from IWB (Russell 1000 ETF)
universe_history = qb.universe_history(
    qb.universe.etf('IWB'), end_date - timedelta(1), end_date
)
symbols = [fundamental.symbol for fundamental in universe_history.iloc[0]]
print(f"Universe size: {len(symbols)} symbols")

# Load 3 years of daily close prices
prices = qb.history(
    symbols, end_date - timedelta(3 * 365), end_date, Resolution.DAILY
)['close'].unstack(0).dropna(axis=1)
print(f"Price data: {prices.shape[0]} days x {prices.shape[1]} symbols")
prices.head()

IndexError: single positional indexer is out-of-bounds

## Step 2: Dimensionality Reduction with PCA

Compute daily returns, standardize them, then reduce to 3 principal components. Each asset is represented by its contribution (loading) to each component, placing it in a 3D coordinate space.

In [3]:
daily_returns = prices.pct_change().dropna()
standardized_returns = StandardScaler().fit_transform(daily_returns)

pca = PCA(n_components=3, random_state=0)
pca.fit(standardized_returns)

factor_exposures = pd.DataFrame(
    index=[f"component_{i}" for i in range(pca.components_.shape[0])],
    columns=daily_returns.columns,
    data=pca.components_
).T

print(f"Explained variance ratios: {pca.explained_variance_ratio_}")
print(f"Total explained: {pca.explained_variance_ratio_.sum():.3f}")
factor_exposures.head(10)

NameError: name 'prices' is not defined

## Step 3: Clustering with OPTICS

Group the assets based on their PCA coordinates using the OPTICS algorithm. Assets in the same cluster tend to have similar return patterns, making them good candidates for pairs trading.

In [4]:
clustering = OPTICS().fit(factor_exposures)

labels = clustering.labels_[clustering.labels_ != -1]
print(
    f"Out of {len(clustering.labels_)} assets, OPTICS found {len(labels)} "
    f"non-noisy samples and organized them into {len(set(labels))} groups."
)

# Show cluster sizes
cluster_counts = pd.Series(clustering.labels_).value_counts().sort_index()
print("\nCluster sizes:")
print(cluster_counts.to_string())

NameError: name 'factor_exposures' is not defined

## Step 4: Pair Validation Tests

For each candidate pair within a cluster, apply 4 tests:
1. **Cointegration** (Engle-Granger, p-value < 0.01)
2. **Hurst exponent** (< 0.5, mean-reverting spread)
3. **Half-life** (between 1 and 252 days)
4. **Mean crossings** (>= 12 per year)

A simplified version uses cointegration and spread stationarity as proxies.

In [5]:
from itertools import combinations
from statsmodels.tsa.stattools import coint
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

pairs_tested = 0
test_results_index = [
    'Failed at cointegration test',
    'Failed at spread stationarity',
    'Failed at half-life test',
]
test_results = pd.DataFrame(
    {"count": [0] * len(test_results_index)},
    index=test_results_index
)
pairs_detected = []

for cluster_id in range(len(set(labels))):
    cluster_symbols = list(
        factor_exposures.index[clustering.labels_ == cluster_id]
    )

    for symbol_a, symbol_b in combinations(cluster_symbols, 2):
        pairs_tested += 1

        # Test 1: Cointegration
        score, pvalue, _ = coint(prices[symbol_a], prices[symbol_b])
        if pvalue > 0.01:
            test_results.loc[test_results_index[0], 'count'] += 1
            continue

        # Compute spread via OLS hedge ratio
        model = OLS(prices[symbol_a], add_constant(prices[symbol_b])).fit()
        spread = prices[symbol_a] - model.params.iloc[1] * prices[symbol_b]

        # Test 2: Spread stationarity (ADF proxy via spread mean-reversion)
        spread_std = spread.std()
        if spread_std <= 0:
            test_results.loc[test_results_index[1], 'count'] += 1
            continue

        # Test 3: Half-life estimation
        lagged_spread = spread.shift(1).dropna()
        spread_delta = (spread - spread.shift(1)).dropna()
        if len(spread_delta) < 10:
            continue
        hl_model = OLS(spread_delta.values, add_constant(lagged_spread.values)).fit()
        beta = hl_model.params[1]
        if beta >= 0:
            test_results.loc[test_results_index[2], 'count'] += 1
            continue
        half_life = -np.log(2) / beta
        if not (1 < half_life < 252):
            test_results.loc[test_results_index[2], 'count'] += 1
            continue

        pairs_detected.append({
            'symbol_a': symbol_a,
            'symbol_b': symbol_b,
            'spread': spread,
            'half_life': half_life,
            'coint_pvalue': pvalue
        })

print(f"Pairs tested: {pairs_tested}")
print(f"Pairs detected: {len(pairs_detected)}")
display(test_results)

NameError: name 'labels' is not defined

## Step 5: Results Summary

Display the detected pairs with their key statistics.

In [6]:
if not pairs_detected:
    print("No pairs detected. Try adjusting the universe or test thresholds.")
else:
    results_df = pd.DataFrame([
        {
            'Symbol A': p['symbol_a'].value,
            'Symbol B': p['symbol_b'].value,
            'Half-life (days)': round(p['half_life'], 1),
            'Coint p-value': f"{p['coint_pvalue']:.4f}",
        }
        for p in pairs_detected
    ])
    print(f"Detected {len(results_df)} trading pairs:")
    display(results_df)

No pairs detected. Try adjusting the universe or test thresholds.
